<a href="https://colab.research.google.com/github/bezawitlegesse/week1-demo/blob/main/week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 Lab: Small Data-Processing App
Pipeline: **load raw data → clean with functions → process with NumPy → report**

In [115]:
%%writefile requirements.txt
ipykernel
numpy
requests

Overwriting requirements.txt


In [116]:
%pip install -q -r requirements.txt

In [117]:

!pip install -q virtualenv
!virtualenv .venv
!.venv/bin/pip install -q -r requirements.txt
!.venv/bin/python -c "import numpy, requests; print('venv OK | numpy', numpy.__version__, '| requests', requests.__version__)"

created virtual environment CPython3.13.16.final.0-64-x86_64 in 891ms
  creator CPython3Posix(dest=/content/.venv, clear=False, no_vcs_ignore=False, venv_redirect=None, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: asttokens==3.0.2, certifi==2026.10.18, charset_normalizer==3.5.2, comm==0.2.3, debugpy==1.8.22, executing==2.3.0, idna==3.20, ipykernel==7.4.0, ipython==9.17.1, ipython_pygments_lexers==1.1.1, jedi==0.20.1, jupyter_client==8.10.0, jupyter_core==5.9.1, matplotlib_inline==0.2.2, nest_asyncio2==1.7.4, numpy==2.5.4, packaging==26.3, parso==0.8.7, pexpect==4.9.0, pip==26.2.1, platformdirs==4.13.5, prompt_toolkit==3.0.53, psutil==7.2.2, ptyprocess==0.7.0, pure_eval==0.2.4, pygments==2.21.0, python_dateutil==2.9.0.post0, pyzmq==27.2.0, requests==2.34.2, six==1.17.0, stack_data==0.6.3, tornado==6.5.10, traitlets==5.16.1, typing_extensions==4.16.0, urllib3==2.8.0, wcwidth==0.9.2
  activators Bas

## 1. Imports and raw dataset
The raw data is messy on purpose: blanks, duplicates, bad numbers, out-of-range scores.

## 2. Load function

In [118]:
import csv
import math
import numpy as np

RAW_CSV = "raw_scores.csv"

raw_rows = """name,score
 Selamawit ,85.5
 Selamawit,85.5
 Hana,0
Bezawit,92
 ,90
Abebe,abc
 Kebede,150
 Selemon,nan
 Chaltu ,
 Beselot, 78
 Hagos,64
 Abale,71.5
Fatuma,88
 Samson,55
 Maron,97
 """

 # Write the raw dataset to a CSV so the notebook runs anywhere (Colab or VS Code)
with open(RAW_CSV, "w", newline="") as f:
     f.write(raw_rows)
     print("Created", RAW_CSV)

Created raw_scores.csv


In [119]:
def load_csv(path):
    """Read a CSV file and return a list of dictionaries (one per row)."""
    with open(path, newline="") as f:
        return list(csv.DictReader(f))

raw_data = load_csv(RAW_CSV)
print(f"Loaded {len(raw_data)} raw rows")
raw_data[:3]

Loaded 16 raw rows


[{'name': ' Selamawit ', 'score': '85.5'},
 {'name': ' Selamawit', 'score': '85.5'},
 {'name': ' Hana', 'score': '0'}]

## 3. Cleaning functions

In [120]:
def parse_score(value):
    """Return a valid score (0-100) as a float, or None if invalid."""
    text = "" if value is None else str(value).strip()
    if not text:
        return None
    try:
        score = float(text)
    except ValueError:
        return None
    if not math.isfinite(score) or score < 0 or score > 100:
        return None
    return score


def clean_data(data):
    """Strip names, drop invalid rows, remove duplicates."""
    cleaned = []
    seen = set()
    for row in data:
        raw_name = row.get("name")
        name = "" if raw_name is None else str(raw_name).strip()
        score = parse_score(row.get("score"))

        if not name or score is None:
            continue

        record = (name, score)
        if record not in seen:
            seen.add(record)
            cleaned.append({"name": name, "score": score})
    return cleaned


clean = clean_data(raw_data)
print(f"Kept {len(clean)} of {len(raw_data)} rows")
for row in clean:
    print(row)

Kept 9 of 16 rows
{'name': 'Selamawit', 'score': 85.5}
{'name': 'Hana', 'score': 0.0}
{'name': 'Bezawit', 'score': 92.0}
{'name': 'Beselot', 'score': 78.0}
{'name': 'Hagos', 'score': 64.0}
{'name': 'Abale', 'score': 71.5}
{'name': 'Fatuma', 'score': 88.0}
{'name': 'Samson', 'score': 55.0}
{'name': 'Maron', 'score': 97.0}


## 4. Process with NumPy

In [121]:
def analyze(clean_rows):
    """Compute statistics, z-scores and letter grades with NumPy."""
    names = np.array([r["name"] for r in clean_rows])
    scores = np.array([r["score"] for r in clean_rows])

    stats = {
        "count": scores.size,
        "mean": scores.mean(),
        "median": np.median(scores),
        "std": scores.std(),
        "min": scores.min(),
        "max": scores.max(),
        "p25": np.percentile(scores, 25),
        "p75": np.percentile(scores, 75),
    }

    z_scores = (scores - stats["mean"]) / stats["std"]

    grades = np.select(
        [scores >= 90, scores >= 80, scores >= 70, scores >= 60],
        ["A", "B", "C", "D"],
        default="F",
    )
    return names, scores, z_scores, grades, stats


names, scores, z_scores, grades, stats = analyze(clean)

## 5. Report

In [122]:
print("=== Summary statistics ===")
for key, value in stats.items():
    print(f"{key:>6}: {value:.2f}")

print("\n=== Results (best first) ===")
order = np.argsort(scores)[::-1]
print(f"{'Name':<8}{'Score':>7}{'Z':>7}  Grade")
for i in order:
    print(f"{names[i]:<8}{scores[i]:>7.1f}{z_scores[i]:>7.2f}  {grades[i]}")

print("\nTop 3:", ", ".join(names[order[:3]]))
print("Above average:", ", ".join(names[scores > stats["mean"]]))

=== Summary statistics ===
 count: 9.00
  mean: 70.11
median: 78.00
   std: 27.90
   min: 0.00
   max: 97.00
   p25: 64.00
   p75: 88.00

=== Results (best first) ===
Name      Score      Z  Grade
Maron      97.0   0.96  A
Bezawit    92.0   0.78  A
Fatuma     88.0   0.64  B
Selamawit   85.5   0.55  B
Beselot    78.0   0.28  C
Abale      71.5   0.05  C
Hagos      64.0  -0.22  D
Samson     55.0  -0.54  F
Hana        0.0  -2.51  F

Top 3: Maron, Bezawit, Fatuma
Above average: Selamawit, Bezawit, Beselot, Abale, Fatuma, Maron
